# Okay is this working? Hello? Guys? HELP ME?

In [1]:
import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt 
from sklearn.model_selection import train_test_split    
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.linear_model import LogisticRegression 
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report 
import requests
from bs4 import BeautifulSoup
import re
import time
from tqdm.notebook import tqdm
from urllib.parse import urlparse
# Load the dataset
data = pd.read_csv('datasets/FakeNewsNet.csv')
#gitignore!!!
import os

In [2]:
df = pd.DataFrame(data)
print(df.head())
print(df.info())


                                               title  \
0  Kandi Burruss Explodes Over Rape Accusation on...   
1  People's Choice Awards 2018: The best red carp...   
2  Sophia Bush Sends Sweet Birthday Message to 'O...   
3  Colombian singer Maluma sparks rumours of inap...   
4  Gossip Girl 10 Years Later: How Upper East Sid...   

                                            news_url        source_domain  \
0  http://toofab.com/2017/05/08/real-housewives-a...           toofab.com   
1  https://www.today.com/style/see-people-s-choic...        www.today.com   
2  https://www.etonline.com/news/220806_sophia_bu...     www.etonline.com   
3  https://www.dailymail.co.uk/news/article-33655...  www.dailymail.co.uk   
4  https://www.zerchoo.com/entertainment/gossip-g...      www.zerchoo.com   

   tweet_num  real  
0         42     1  
1          0     1  
2         63     1  
3         20     1  
4         38     1  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23196 entries, 0 to 2319

In [3]:
# not nulls
print(df.isnull().sum())

title              0
news_url         330
source_domain    330
tweet_num          0
real               0
dtype: int64


# Prepare for scraping

In [4]:
print(df.columns.tolist())

URL_COL = "news_url"
assert URL_COL in df.columns, f"Column '{URL_COL}' not found in CSV"

['title', 'news_url', 'source_domain', 'tweet_num', 'real']


##

In [5]:
# Beautiful Soup Template https://www.crummy.com/software/BeautifulSoup/bs4/doc/#quick-start

# User-Agent header to mimic a real browser and avoid blocking, can be expanded to rotate or use proxies if needed. Whatever man.
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    )
}

# Helper functions for text cleaning and word extraction
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def extract_words(text):
    if not isinstance(text, str):
        return []
    return re.findall(r"\b[a-zA-Z]+\b", text.lower())





#==================================== Main scraping function with error handling and smart content extraction===================================================
def scrape_website(url, timeout=10):
    if pd.isna(url) or not isinstance(url, str) or not url.strip(): #shits busted
        return "", "", 0, "missing_url" # make sure to log that we are missing urls
    try:
        response = requests.get(url, headers=HEADERS, timeout=timeout)
        response.raise_for_status() # will raise an HTTPError for bad responses (4xx and 5xx) "not found" and "server error" and all that jizz
        soup = BeautifulSoup(response.text, "html.parser") # gets rid of all the shit we dont need and makes it easier to extract the text



#======================================= ---JUNK WORDS REMOVAL--- ========================================================
        for tag in soup(["script", "style", "noscript", "header", "footer", "nav", "aside"]):
            tag.decompose() # execute it



        # Smart Article Extraction: Try common tags first, then fallback to body or whole text
        content = soup.find("article")
        if content is None:
            content = soup.find("main")
        if content is None:
            content = soup.body
        if content is None:
            content = soup

        text = content.get_text(separator=" ")
        text = clean_text(text) # helper frunction from above

        # hehe frunction

        words = extract_words(text)
        words_joined = " ".join(words)

        return text, words_joined, len(words), "success"

    # fun fact not havbing these crashes your entire trace and you lose all your progress!
    # LOG
    # YOUR
    # SHIT
    except requests.exceptions.RequestException as e:
        return "", "", 0, f"request_error: {str(e)}"
    except Exception as e:
        return "", "", 0, f"parse_error: {str(e)}"

# URL Normalization
- Beautsoup dies if there is no https:// so i found a libnrtary that does that

In [6]:
# sanity check
from url_normalize import url_normalize

test_urls = [
    "www.cnn.com",
    "cnn.com",
    "foxnews.com/article/123",
    "https://www.nytimes.com",
    "http://www.bbc.com/news",
    "//reuters.com/world",
    "www.example.com/test",
    "https://https://abcnews.go.com",
    "www.wsj.com/articles/something"
]

for u in test_urls:
    print(u, " -> ", url_normalize(u))

# HYPE

www.cnn.com  ->  https://www.cnn.com/
cnn.com  ->  https://cnn.com/
foxnews.com/article/123  ->  https://foxnews.com/article/123
https://www.nytimes.com  ->  https://www.nytimes.com/
http://www.bbc.com/news  ->  http://www.bbc.com/news
//reuters.com/world  ->  https://reuters.com/world
www.example.com/test  ->  https://www.example.com/test
https://https://abcnews.go.com  ->  https://https/abcnews.go.com
www.wsj.com/articles/something  ->  https://www.wsj.com/articles/something


## Scrape Once.

In [7]:

test_url = df.loc[0, URL_COL]
print("Testing:", test_url)

text, words, count, status = scrape_website(test_url)
print("Status:", status)
print("Word count:", count)
print(text[:500])

# cool

Testing: http://toofab.com/2017/05/08/real-housewives-atlanta-kandi-burruss-rape-phaedra-parks-porsha-williams/
Status: success
Word count: 11
Shannon Elizabeth, Who Played Nadia in 'American Pie,' to Join OnlyFans


# John Ryan Scrape

In [8]:
# Fragnum Opus: The Freat Frcrape

# ========================================================== Settings===========================================================
ORIGINAL_FILE = "datasets/FakeNewsNet.csv"
OUTPUT_FILE = "FakeNewsNet_scraped.csv"
URL_COL = "news_url"
CHECKPOINT_EVERY = 100 # print progress every N rows, actual writing happens every row
SLEEP_SECONDS = 0.8    # be kiond to servers, adjust as needed, lowering value might speed up but increases risk of being blocked
START_AT = 0           # auto-resume will update this

# ========================================================== Load Data ============================================================

# figure out how many rows already exist in the output file so we can resume safely
if os.path.exists(OUTPUT_FILE):
    completed_df = pd.read_csv(OUTPUT_FILE)
    START_AT = len(completed_df)
    print(f"Loaded checkpoint file: {OUTPUT_FILE}")
    print(f"Already scraped rows: {START_AT}")
else:
    completed_df = None
    print(f"No checkpoint found. Starting fresh from: {ORIGINAL_FILE}")

# load original source CSV
source_df = pd.read_csv(ORIGINAL_FILE)
print(f"Total source rows: {len(source_df)}")

# ========================================================== Main Loop ============================================================

# if output file does not exist yet, create it with the full header first
if not os.path.exists(OUTPUT_FILE):
    first_chunk = source_df.iloc[:0].copy()
    first_chunk["fixed_url"] = None
    first_chunk["scraped_text"] = None
    first_chunk["scraped_words"] = None
    first_chunk["scraped_word_count"] = None
    first_chunk["scrape_status"] = None
    first_chunk.to_csv(OUTPUT_FILE, index=False)

#===============================MAIN LOOP WITH PROGRESS BAR AND APPENDING============================================================
# really cool progress bar with tqdm i found, it shows progress and estimated time remaining, super helpful for long scrapes
for i in tqdm(range(START_AT, len(source_df)), desc="Scraping websites >:3"):

    # grab original row
    row = source_df.iloc[i].copy()

    # URL Normalization: Clean and standardize URLs to improve scraping success rates, handles missing schemes, trailing slashes, etc.
    fixed_url = url_normalize(row[URL_COL]) if isinstance(row[URL_COL], str) and row[URL_COL].strip() else None

    # Mainline BSOUP scraping logic, call our helper function and store results in the row
    text, words, count, status = scrape_website(fixed_url)

    row["fixed_url"] = fixed_url
    row["scraped_text"] = text
    row["scraped_words"] = words
    row["scraped_word_count"] = count
    row["scrape_status"] = status

    # write exactly one completed row to the output CSV immediately so my RAM does not get cooked!!!!!
    """
    Let me tell you a story about a deeply unqualified programmer who thought it was somehow reasonable to store every scraped article in a DataFrame and then rewrite the entire CSV every single loop for 23,000 rows.
    Unsurprisingly, that was a horrible idea.
    The program was slow as hell, memory usage kept climbing, and eventually it would just crash because apparently making the machine rewrite the whole file thousands of f***ing times is not "efficient", no matter how much you believe in it.
    I finally stopped being stupid and changed it so it appends new rows to the CSV instead of rebuilding the whole thing every time. That immediately used way less memory and the crashes stopped.
    Stay in school kids.
    """
    # only add to csv if extraction was successful
    if(status == "success"):
        pd.DataFrame([row]).to_csv(OUTPUT_FILE, mode="a", header=False, index=False)

    """ # checkpointing: row is already saved every iteration, this is just a status print now. Redundant
    if (i + 1) % CHECKPOINT_EVERY == 0:
        print(f"Checkpoint Saved at row {i+1}.")
    """

    # john status update for each row, helps track progress and debug issues with specific URLs
    print(f"Row {i+1}/{len(source_df)}: URL='{fixed_url}' Status='{status}' Word Count={count}")

    time.sleep(SLEEP_SECONDS)

print(f"Yo Chud. Im Done. Saved to {OUTPUT_FILE}")

Loaded checkpoint file: FakeNewsNet_scraped.csv
Already scraped rows: 7
Total source rows: 23196


Scraping websites >:3:   0%|          | 0/23189 [00:00<?, ?it/s]

Row 8/23196: URL='https://www.etonline.com/news/214798_amber_rose_shuts_down_french_montana_dating_rumors' Status='success' Word Count=480
Row 9/23196: URL='https://www.aol.com/article/entertainment/2018/02/23/mindy-kaling-makes-first-post-baby-appearance-at-disneyland-with-her-wrinkle-in-time-co-stars/23369408/' Status='request_error: 404 Client Error: Not Found for url: https://www.aol.com/article/entertainment/2018/02/23/mindy-kaling-makes-first-post-baby-appearance-at-disneyland-with-her-wrinkle-in-time-co-stars/23369408/' Word Count=0
Row 10/23196: URL='https://www.98online.com/2018/05/02/katharine-mcphee-butchers-tony-nominations-i-have-not-been-drinking/' Status='request_error: 404 Client Error: Not Found for url: https://www.98online.com/2018/05/02/katharine-mcphee-butchers-tony-nominations-i-have-not-been-drinking/' Word Count=0
Row 11/23196: URL='https://www.usmagazine.com/celebrity-news/news/wags-miami-stars-ashley-nicole-roberts-and-philip-wheeler-are-married-w482173/' Stat

KeyboardInterrupt: 

# NLP Preprocessing

In [12]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

df = pd.read_csv('FakeNewsNet_scraped.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   title               9 non-null      object
 1   news_url            9 non-null      object
 2   source_domain       9 non-null      object
 3   tweet_num           9 non-null      int64 
 4   real                9 non-null      int64 
 5   fixed_url           9 non-null      object
 6   scraped_text        9 non-null      object
 7   scraped_words       9 non-null      object
 8   scraped_word_count  9 non-null      int64 
 9   scrape_status       9 non-null      object
dtypes: int64(3), object(7)
memory usage: 848.0+ bytes


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jack8\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\jack8\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\jack8\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
import re



lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):

  text = text.lower()

  text = re.sub(r'httpS+', '', text)
  text = re.sub(r'<.*?>', '', text)
  text = re.sub(r'[^a-zs]', '', text)

  tokens = text.split()
  tokens = {
    lemmatizer.lematize(w) for w in tokens
    if w not in stop_words and len(w) > 2
  }

  return ' '.join(tokens)